# Spoken Grammar Scoring — Baseline
Author: Shruti Suman. SHL Research Intern hiring assessment.

This notebook maps an audio recording to a continuous grammar score in [0, 5].
The current baseline uses frozen Whisper Turbo transcription, word/character
TF-IDF features, and Ridge regression. No external labelled dataset is used.

**Status:** baseline, not the final solution. This reorganized public copy has
cleared execution outputs. The report contains actual aggregate results supplied
from the author's earlier Kaggle run on 5 October 2026; this copy has not been
rerun against the private data. Reproduce and update it before final submission.

Keep the audio, source label CSVs, transcripts, embeddings, fitted models and
record-level predictions private. Download competition data only through Kaggle
after accepting its rules. Do not commit them to the public repository.


## 1. Environment
Use a private Kaggle notebook with the competition dataset attached. Internet
access is needed for package/model downloads. A GPU speeds up ASR; CPU int8 is
supported but slower. The original run used a Tesla T4 and PyAV 18.1.0.
The PyAV pin avoids the observed `metadata_errors` decoder incompatibility.
On first install, restart the kernel if already-imported libraries are changed,
then continue from the next cell. Do not repeatedly restart after every cell.


In [ ]:
%pip install -q faster-whisper "av==18.1.0" pandas numpy scipy scikit-learn matplotlib joblib


In [ ]:
from pathlib import Path
import importlib.metadata as package_metadata
import json
import re
import time

import av
import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import pearsonr
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import mean_squared_error

output_dir = Path('/kaggle/working')
output_dir.mkdir(parents=True, exist_ok=True)
print('PyAV:', av.__version__, '| GPU:', torch.cuda.is_available())
versions = {}
for package in ['faster-whisper', 'av', 'torch', 'pandas', 'numpy',
                'scipy', 'scikit-learn', 'matplotlib', 'joblib']:
    versions[package] = package_metadata.version(package)
(output_dir / 'environment_versions.json').write_text(json.dumps(versions, indent=2))


## 2. Data loading and checks
The train labels are used as supplied, including zeros. Test labels are random
placeholders and are discarded. Index audio separately within train and test:
the same basename can refer to different recordings. CSV joins are validated.
The sample submission mismatch must be resolved before an official submission;
do not fabricate missing filenames or change labels to address it.


In [ ]:
def load_csv(name):
    matches = list(Path('/kaggle/input').rglob(name))
    if len(matches) != 1:
        raise ValueError(f'Expected one {name}, found {len(matches)}')
    return pd.read_csv(matches[0], dtype={'filename': str}), matches[0].parent

train_df, data_dir = load_csv('train.csv')
test_df, _ = load_csv('test.csv')
sample_df, _ = load_csv('sample_submission.csv')
test_df = test_df[['filename']].copy()

def attach_audio_paths(df, split):
    index = {}
    for path in (data_dir / split).rglob('*'):
        if path.is_file() and path.suffix.lower() == '.wav':
            if path.name in index:
                raise ValueError(f'Duplicate audio basename in {split}')
            index[path.name] = path
    names = df['filename'].map(lambda value: Path(value.replace('\\', '/')).name)
    if not names.isin(index).all():
        raise FileNotFoundError(f'{split}: CSV audio missing')
    df['audio_path'] = names.map(lambda name: str(index[name]))

for frame in [train_df, test_df, sample_df]:
    assert not frame['filename'].duplicated().any()
    assert not frame.isna().any().any()
assert train_df['label'].between(0, 5).all()
attach_audio_paths(train_df, 'train')
attach_audio_paths(test_df, 'test')

submission_ids_match = set(test_df['filename']) == set(sample_df['filename'])
print('Train:', len(train_df), '| Test:', len(test_df), '| Sample:', len(sample_df))
print('Sample IDs match test IDs:', submission_ids_match)
print('Test IDs absent from sample:', len(set(test_df['filename']) - set(sample_df['filename'])))
print('Sample IDs absent from test:', len(set(sample_df['filename']) - set(test_df['filename'])))


## 3. Audio-to-text preprocessing
Whisper Turbo transcribes the audio locally. Automatic language detection is
retained; VAD filters silence and previous-text conditioning is disabled.
Do not correct transcript grammar: errors are relevant to the target.
Whisper internally handles audio decoding/resampling and inference windows.
Checkpoint train and test separately, saving every ten attempts and on exit.
Completed recordings are skipped on rerun. A successful empty transcript is
flagged at the modeling stage, not silently converted to a zero score.
Inspect a few training examples privately for ASR quality, without manually
assigning replacement labels. Checkpoints persist only while their files remain
available; download them privately before stopping the session.


In [ ]:
settings = dict(model='turbo', task='transcribe', language='auto',
                beam_size=5, vad_filter=True, condition_on_previous_text=False)
settings_path = output_dir / 'transcription_settings.json'
if settings_path.exists():
    if json.loads(settings_path.read_text()) != settings:
        raise ValueError('Existing checkpoints use different ASR settings')
else:
    settings_path.write_text(json.dumps(settings, indent=2))

columns = ['filename', 'transcript', 'language', 'audio_seconds',
           'processing_seconds', 'word_count', 'status', 'error']
asr_model = None

def save_checkpoint(records, path):
    temporary = path.with_suffix('.tmp')
    pd.DataFrame(list(records.values()), columns=columns).to_csv(temporary, index=False)
    temporary.replace(path)

def transcribe_split(df, split):
    global asr_model
    path = output_dir / f'{split}_transcripts.csv'
    records = {}
    if path.exists():
        cached = pd.read_csv(path, keep_default_na=False, dtype={'filename': str})
        assert not cached['filename'].duplicated().any()
        records = {item['filename']: item for item in cached.to_dict('records')}
    attempted = 0
    try:
        for _, row in df.iterrows():
            name = str(row['filename'])
            if records.get(name, {}).get('status') == 'ok':
                continue
            if asr_model is None:
                from faster_whisper import WhisperModel
                device = 'cuda' if torch.cuda.is_available() else 'cpu'
                compute_type = 'float16' if device == 'cuda' else 'int8'
                asr_model = WhisperModel('turbo', device=device, device_index=0,
                                         compute_type=compute_type)
            started = time.perf_counter()
            try:
                segments, info = asr_model.transcribe(row['audio_path'],
                    task='transcribe', beam_size=5, vad_filter=True,
                    condition_on_previous_text=False)
                text = ' '.join(segment.text.strip() for segment in segments).strip()
                records[name] = dict(filename=name, transcript=text,
                    language=info.language, audio_seconds=info.duration,
                    processing_seconds=time.perf_counter() - started,
                    word_count=len(text.split()), status='ok', error='')
            except Exception as exc:
                records[name] = dict(filename=name, transcript='', language='',
                    audio_seconds='', processing_seconds=time.perf_counter() - started,
                    word_count=0, status='error', error=f'{type(exc).__name__}: {exc}')
                print(f'{split}: a recording failed; inspect private checkpoint')
            attempted += 1
            if attempted % 10 == 0:
                save_checkpoint(records, path)
            if attempted % 25 == 0:
                print(f'{split}: attempted {attempted} new recordings', flush=True)
    finally:
        save_checkpoint(records, path)
    return pd.DataFrame([records[name] for name in df['filename']], columns=columns)

train_text = transcribe_split(train_df, 'train')
test_text = transcribe_split(test_df, 'test')
summary = []
for split, frame in [('train', train_text), ('test', test_text)]:
    ok = frame['status'].eq('ok')
    summary.append(dict(split=split, total=len(frame), completed=int(ok.sum()),
        failed=int((~ok).sum()), empty_transcripts=int((ok & frame['transcript'].str.strip().eq('')).sum())))
display(pd.DataFrame(summary))


## 4. Pipeline architecture and evaluation
Audio → frozen ASR → uncorrected transcript → word TF-IDF (1–2 grams) and
character TF-IDF (3–5 grams) → Ridge (alpha=1) → clipping to [0, 5].

Keep function words; they can convey grammatical usage. TF-IDF vocabularies and
IDF weights are learned only from each fold's training examples. Five shuffled
folds use seed 42. If exact normalized transcripts repeat, GroupKFold keeps
them together. This grouping does not identify all shared speakers or prompts.
No hyperparameter tuning is performed for this baseline.

RMSE measures score error; Pearson measures linear association. Out-of-fold
(OOF) predictions estimate held-out performance. In-sample training RMSE is
reported separately to meet the assignment requirement and is not treated as
an estimate of generalization. Pearson is undefined for constant predictions.


In [ ]:
def merge_text(source, text):
    assert set(source['filename']) == set(text['filename'])
    result = source.merge(text, on='filename', how='left', validate='one_to_one', sort=False)
    assert result['status'].eq('ok').all(), 'Resolve ASR failures first'
    assert result['transcript'].str.strip().ne('').all(), 'Inspect empty transcripts first'
    return result

training = merge_text(train_df[['filename', 'label']], train_text)
testing = merge_text(test_df[['filename']], test_text)
texts = training['transcript'].astype(str)
y = training['label'].to_numpy(dtype=float)

pipeline = Pipeline([
    ('features', FeatureUnion([
        ('word', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2,
            max_features=25000, sublinear_tf=True, strip_accents='unicode')),
        ('character', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2,
            max_features=40000, sublinear_tf=True, strip_accents='unicode')),
    ])),
    ('regressor', Ridge(alpha=1.0, solver='lsqr')),
])

def rmse(actual, predicted):
    return float(np.sqrt(mean_squared_error(actual, predicted)))

def correlation(actual, predicted):
    if np.std(actual) < 1e-12 or np.std(predicted) < 1e-12:
        return np.nan
    return float(pearsonr(actual, predicted)[0])

normalized = texts.map(lambda value: re.sub(r'\s+', ' ', value.lower()).strip())
groups, _ = pd.factorize(normalized)
if normalized.duplicated().any():
    folds = list(GroupKFold(n_splits=5).split(texts, y, groups=groups))
else:
    folds = list(KFold(n_splits=5, shuffle=True, random_state=42).split(texts))

oof = np.full(len(y), np.nan)
mean_oof = np.full(len(y), np.nan)
fold_ids = np.zeros(len(y), dtype=int)
fold_results = []
for fold, (fit_ids, valid_ids) in enumerate(folds, 1):
    print(f'Training fold {fold}/5', flush=True)
    model = clone(pipeline).fit(texts.iloc[fit_ids], y[fit_ids])
    pred = np.clip(model.predict(texts.iloc[valid_ids]), 0, 5)
    constant = np.full(len(valid_ids), y[fit_ids].mean())
    oof[valid_ids], mean_oof[valid_ids], fold_ids[valid_ids] = pred, constant, fold
    fold_results.append(dict(fold=fold, validation_samples=len(valid_ids),
        mean_baseline_RMSE=rmse(y[valid_ids], constant), model_RMSE=rmse(y[valid_ids], pred),
        model_Pearson=correlation(y[valid_ids], pred)))
assert np.isfinite(oof).all()
display(pd.DataFrame(fold_results).round(4))

final_model = clone(pipeline).fit(texts, y)
train_pred = np.clip(final_model.predict(texts), 0, 5)
test_pred = np.clip(final_model.predict(testing['transcript']), 0, 5)
metrics = pd.DataFrame([
    dict(evaluation='Mean baseline — out of fold', RMSE=rmse(y, mean_oof), Pearson=correlation(y, mean_oof)),
    dict(evaluation='TF-IDF + Ridge — out of fold', RMSE=rmse(y, oof), Pearson=correlation(y, oof)),
    dict(evaluation='TF-IDF + Ridge — training fit', RMSE=rmse(y, train_pred), Pearson=correlation(y, train_pred)),
])
display(metrics.round(4))
metrics.to_csv(output_dir / 'baseline_metrics.csv', index=False)
pd.DataFrame(fold_results).to_csv(output_dir / 'baseline_fold_metrics.csv', index=False)
pd.DataFrame(dict(filename=training['filename'], label=y, fold=fold_ids,
    oof_prediction=oof, training_fit_prediction=train_pred)).to_csv(output_dir / 'baseline_oof_predictions.csv', index=False)
pd.DataFrame(dict(filename=testing['filename'], label=test_pred)).to_csv(output_dir / 'baseline_test_predictions.csv', index=False)
joblib.dump(final_model, output_dir / 'tfidf_ridge_baseline.joblib')


## 5. Visualizations and error analysis
Use held-out predictions for the scatter plot and residual histogram. Break
down errors by actual training score. Do not remove zero-labelled rows based
on their poor performance. Any clarification about the meaning of zero labels
must come from the organizers, not a guessed replacement label.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y, oof, alpha=0.4, s=20)
axes[0].plot([0, 5], [0, 5], 'r--', linewidth=1)
axes[0].set(xlabel='Actual grammar score', ylabel='Out-of-fold predicted score',
            title='Held-out predictions', xlim=(-0.1, 5.1), ylim=(-0.1, 5.1))
axes[1].hist(oof - y, bins=25, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set(xlabel='Predicted score minus actual score', ylabel='Number of recordings',
            title='Held-out prediction errors')
fig.tight_layout()
fig.savefig(output_dir / 'baseline_validation.png', dpi=180, bbox_inches='tight')
plt.show()

diagnostics = pd.DataFrame(dict(label=y, prediction=oof, squared_error=(oof-y)**2))
by_score = diagnostics.groupby('label').agg(recordings=('label', 'size'),
    mean_prediction=('prediction', 'mean'), mean_squared_error=('squared_error', 'mean'))
by_score['mean_error'] = by_score['mean_prediction'] - by_score.index
by_score['RMSE'] = np.sqrt(by_score.pop('mean_squared_error'))
display(by_score.round(3))
by_score.to_csv(output_dir / 'baseline_errors_by_score.csv')


## 6. Brief report — observed baseline run

**Data and preprocessing.** All 769 training and 216 test recordings were
transcribed successfully with no empty transcripts. Detected languages were
English for 768 training clips and all test clips, with one training clip
detected as Indonesian; language detection can be wrong. Train and test CSVs
have no duplicate filenames within a split. Of 212 shared basenames across
splits, none of the paired files were byte-identical. This does not prove the
absence of all duplicate recordings or shared speakers. Zero labels were kept.

**Method.** Frozen Whisper Turbo with beam size 5 and VAD, followed by word and
character TF-IDF and Ridge regression (alpha=1, LSQR solver). Predictions were
clipped to [0, 5]. Test placeholder labels were ignored. The original run had
no exact normalized transcript duplicates, so five shuffled folds with seed 42
were used. Features were fitted within each fold.

**Evaluation results (actual earlier Kaggle run; not invented cell outputs).**

| Evaluation | RMSE | Pearson |
|---|---:|---:|
| Mean baseline, out of fold | 1.2414 | -0.0961 |
| TF-IDF + Ridge, out of fold | 1.0901 | 0.4746 |
| TF-IDF + Ridge, training fit | **0.4811** | 0.9567 |

The compulsory **training RMSE is 0.4811** for this baseline. The OOF RMSE of
1.0901 is about 12.2% lower than the mean baseline. The mean model predicts
each training fold's mean; its pooled OOF predictions therefore vary slightly,
so the small negative pooled Pearson is not meaningful evidence of skill.
The large training/validation gap suggests overfitting. Predicted scores are
compressed toward the middle. The 37 zero-labelled recordings (4.8% of train)
account for approximately 45.0% of squared OOF error; their RMSE is 3.333.

**Limitations.** ASR can alter mistakes or hallucinate text. TF-IDF can learn
topic associations rather than grammatical control. Random folds may be
optimistic if speakers or prompts are shared; no reliable speaker metadata has
been supplied. The SHL threshold and leaderboard performance are unknown.

**Submission blocker.** The sample has 204 rows while test has 216; only 25
filenames match. There are 191 test IDs absent from the sample and 179 sample
IDs absent from test, including after basename/stem checks. The saved 216-row
prediction file is provisional. No official submission is claimed here.

**Next experiment.** Frozen MPNet embeddings with nested selection of Ridge
regularization, evaluated on the same outer folds. Results are pending and
are not claimed by this baseline notebook.


## 7. Submission and reproducibility
Run this notebook in a private competition notebook with the original dataset
attached. Download checkpoints privately. Before final handoff, rerun the final
chosen pipeline, update its training and held-out metrics, resolve submission
identifiers, and submit the valid output through Kaggle. Supply the mandatory
SHL form with the actual Kaggle username, actual submission information, public
repository URL, and requested final output file. GitHub publication alone does
not complete the hiring challenge.

Public copies must contain code, report and aggregate results only. Share the
same competition code through the competition-associated Kaggle notebook or
discussion as required by the supplied rules. No record-level outputs or
competition-derived files should be included in a public copy.

References: [faster-whisper](https://github.com/SYSTRAN/faster-whisper),
[Whisper](https://github.com/openai/whisper),
[scikit-learn leakage guidance](https://scikit-learn.org/stable/common_pitfalls.html),
[Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html).
AI assistance was used for implementation and debugging. The author should be
able to explain preprocessing, leakage prevention, regularization and metrics.
